# CPU Energy Correction: RAPL Counter Overflow

This notebook corrects the **CPU and DRAM energy** recorded by the main experiment (`results/main`). The **GPU energy is not affected**: it was measured correctly for every inference and is never modified here.

The version of `evaluate_from_local.py` used to run the experiment handled the overflow of the RAPL counters with the wrong counter size, so long inferences lost part of their CPU energy. The lost energy can be reconstructed exactly, which is what this notebook does:

1. Load each run and restore the originally recorded values
2. Reconstruct the energy lost to the counter overflow
3. Recompute the summary of each run
4. Write the per-prompt records, the `run_data.json` files and the `run_table.csv`
5. Validate the results

**The notebook can be run any number of times.** The originally recorded values are kept inside the corrected records, so every execution starts again from them, regardless of whether the data on disk has already been corrected.

The problem is fixed in the current version of `evaluate_from_local.py`. Runs produced by it do not need this correction.

> **Not corrected here: the second processor.** The server has two processors, and the version of the script used in the experiment only read the RAPL counters of the first one. The energy of the second processor was never measured, and no attempt is made to estimate it. The CPU and DRAM values of `results/main`, and the totals that include them, therefore cover the **first processor only**. The current version of the script reads both.

## 1. Setup

With `APPLY = False`, the notebook computes everything and reports what would change, without writing anything.

In [1]:
import csv
import json
import math
import statistics
from pathlib import Path

import pandas as pd

ROOT_DIR = Path("..").resolve()

# Results of the main experiment, corrected in place
RESULTS_DIR = ROOT_DIR / "results/main"
# Write the corrected values to disk
APPLY = True

# Size the script wrongly assumed for the RAPL counters (µJ)
M = 2**32
# Actual size of the RAPL counters of the server (µJ), from /sys/class/powercap/intel-rapl:*/max_energy_range_uj
MAX_ENERGY_RANGE_UJ = 262143328850

# Measured fields of each per-prompt record
CPU_FIELD = "inference_cpu_joules"
DRAM_FIELD = "inference_dram_joules"
GPU_FIELD = "inference_gpu_joules"
TOTAL_FIELD = "total_inference_joules"
TIME_FIELD = "inference_time_seconds"

# Field added by the correction: the originally recorded value and what was restored
CORRECTION_FIELD = "rapl_correction"

## 2. Loading the runs and restoring the recorded values

Each run directory contains one JSON file per MMLU-Pro subject, with one record per question, and a `run_data.json` file with the summary of the run.

Before applying the correction, every record is brought back to the state in which it was originally recorded. The correction changes `inference_cpu_joules` and `inference_dram_joules` in place, and keeps the value originally recorded in the `rapl_correction` field of the record (`stored_joules`). Records with this field are restored from it. Records without it were never changed.

This is what makes the notebook safe to run more than once.

In [2]:
def run_directories():
    directories = [d for d in RESULTS_DIR.glob("run_*") if d.is_dir()]
    # run_<model index>_repetition_<repetition index>
    return sorted(directories, key=lambda d: tuple(int(x) for x in d.name.split("_")[1::2]))


def load_run(run_dir):
    # Returns {file path: records} with the per-prompt records of a run, and its summary
    files = {}
    for path in sorted(run_dir.glob("*/CoT/all/*.json")):
        with open(path, "r") as f:
            files[path] = json.load(f)
    with open(run_dir / "run_data.json", "r") as f:
        run_data = json.load(f)
    return files, run_data


def restore_recorded_values(record):
    correction = record.pop(CORRECTION_FIELD, {})
    if "cpu" in correction:
        record[CPU_FIELD] = correction["cpu"]["stored_joules"]
    if "dram" in correction:
        record[DRAM_FIELD] = correction["dram"]["stored_joules"]


    record[TOTAL_FIELD] = record[CPU_FIELD] + record[DRAM_FIELD] + record[GPU_FIELD]


print(f"Runs found: {len(run_directories())}")

Runs found: 530


## 3. The correction

### The problem

The energy of an inference is the difference between two readings of a counter that only goes up. These counters have a limited size: when they reach their maximum, $R$, they start again from zero. On this server, $R = 262{,}143{,}328{,}850$ µJ (about 262 kJ).

The script tried to handle this by computing the difference as `(end - start) % 2**32`, assuming the counters had $M = 2^{32}$ µJ (about 4,295 J) of range. Since that is not their actual size, two things went wrong:

- **Long inferences were truncated.** Any inference that used more than 4,295 J lost one or more multiples of $M$. At the CPU's usual 87 W, this affects every inference longer than about 50 seconds.
- **Real overflows were corrected by the wrong amount.**

### What can be recovered

For a recorded value $s$, the true energy is one of:

$$\text{true} = s + k \cdot M \qquad \text{(the counter did not overflow during the inference)}$$
$$\text{true} = s + (R \bmod M) + k \cdot M \qquad \text{(the counter overflowed once during the inference)}$$

where $k$ is the number of lost multiples of $M$.

**Finding $k$.** Within a run, the CPU power is very stable (it varies by about 1%). The energy of an inference should therefore be close to the run's usual power multiplied by the inference time. Since candidates with different $k$ are 4,295 J apart, the one closest to that expected value is unambiguous.

**The overflow case cannot be identified in general.** Its candidates are only $R \bmod M \approx 150$ J away from the others, which is within the normal variation between inferences. It is only assumed when the recorded value is impossible otherwise: a short inference recorded with more than 4,000 J. This happens when the counter overflows during an inference that used less than 150 J. Overflows that go undetected leave the value about 150 J too low, at most once for every 262 kJ consumed.

### The reference power of a run

The usual power of a run cannot be taken as a simple median of energy divided by time, because in some runs most of the recorded values are truncated. Instead, the reference power is the one that agrees with the largest number of recorded values *modulo* $M$: a value $s$ recorded in time $t$ agrees with a power $P$ when $P \cdot t$ is close to $s + k \cdot M$ for some $k$.

In [3]:
R_MOD_M = MAX_ENERGY_RANGE_UJ % M


def reference_power(records, field):
    # Recorded energy (µJ) and time of every inference long enough to give a stable power reading
    timed = [(r[field] * 1e6, r[TIME_FIELD]) for r in records if r[TIME_FIELD] >= 1]

    def distance(s, t, power):
        d = (power * t * 1e6 - s) % M
        return min(d, M - d)

    def agreeing(power):
        return [(s, t) for s, t in timed if distance(s, t, power) < 0.05 * power * t * 1e6 + 20e6]

    # Candidate powers: the ones given by the recorded values themselves, rounded to 0.5 W
    candidates = {round(s / t / 1e6 * 2) / 2 for s, t in timed if s > 0}
    best = max(candidates, key=lambda power: len(agreeing(power)))

    # Median power of the agreeing values, after restoring their lost multiples of M
    return statistics.median((s + round((best * t * 1e6 - s) / M) * M) / t / 1e6 for s, t in agreeing(best))


def correct_overflow(recorded_joules, time_seconds, power):
    s = round(recorded_joules * 1e6)
    expected = power * time_seconds * 1e6

    # The counter did not overflow: restore the k lost multiples of M
    k = max(0, round((expected - s) / M))
    value, overflowed = s + k * M, False
    error = abs(value - expected)

    # The counter overflowed during an inference of less than R mod M: the recorded value is about M too high
    overflow_value = s + R_MOD_M - M
    if error > 0.5 * M and overflow_value >= 0 and abs(overflow_value - expected) < 0.1 * error:
        value, overflowed, k = overflow_value, True, -1
        error = abs(value - expected)

    return {
        "joules": value / 1e6,
        "k": k,
        "overflowed": overflowed,
        "expected_joules": expected / 1e6,
        # Distance to the expected value, as a fraction of M. A wrong k would be about 1.
        "error_fraction": error / M,
    }


def apply_overflow_correction(records):
    # Corrects the CPU and DRAM energy of every record of a run. Returns the corrections made.
    log = []
    powers = {"cpu": reference_power(records, CPU_FIELD), "dram": reference_power(records, DRAM_FIELD)}

    for record in records:
        correction = {}
        for domain, field in (("cpu", CPU_FIELD), ("dram", DRAM_FIELD)):
            result = correct_overflow(record[field], record[TIME_FIELD], powers[domain])
            if result["k"] != 0:
                correction[domain] = {"stored_joules": record[field], "wrapped": int(result["overflowed"]), "k": result["k"]}
                log.append({
                    "question_id": record["question_id"],
                    "domain": domain,
                    "inference_time_seconds": record[TIME_FIELD],
                    "stored_joules": record[field],
                    "corrected_joules": result["joules"],
                    "expected_joules": round(result["expected_joules"], 3),
                    "wrapped": int(result["overflowed"]),
                    "k": result["k"],
                    "error_fraction": round(result["error_fraction"], 3),
                })
                record[field] = result["joules"]

        if correction:
            record[TOTAL_FIELD] = record[CPU_FIELD] + record[DRAM_FIELD] + record[GPU_FIELD]
            # Keeps the originally recorded values, so the correction can be traced and reverted
            record[CORRECTION_FIELD] = correction

    return log, powers

## 4. Recomputing the summary of each run

The `run_data.json` file of a run, and its row in `run_table.csv`, are computed from its per-prompt records, in the same way as `evaluate_from_local.py` does:

- The efficiency metrics based on the **total measured energy** (GPU, first processor and its DRAM) keep their names (`accuracy_per_joule`, `tokens_per_joule`, `joules_per_token`, `avg_joules_per_inference`).
- The same metrics are added for the **GPU energy** alone (`accuracy_per_gpu_joule`, `tokens_per_gpu_joule`, `gpu_joules_per_token`, `avg_gpu_joules_per_inference`). These only depend on values that were measured directly and needed no correction, and are the ones used in the analysis.

In [4]:
def summarize_run(records, run_data):
    n = len(records)
    cpu = sum(r[CPU_FIELD] for r in records)
    dram = sum(r[DRAM_FIELD] for r in records)
    gpu = sum(r[GPU_FIELD] for r in records)
    total = sum(r[TOTAL_FIELD] for r in records)
    tokens = sum(r["total_tokens"] for r in records)
    time_seconds = sum(r[TIME_FIELD] for r in records)
    accuracy = run_data["avg_accuracy"]

    return {
        # Efficiency on the total measured energy
        "accuracy_per_joule": accuracy / (total / n),
        "tokens_per_joule": tokens / total,
        "joules_per_token": total / tokens,
        "avg_joules_per_inference": total / n,
        # Energy
        "total_inference_joules": total,
        "total_inference_cpu_joules": cpu,
        "total_inference_dram_joules": dram,
        "total_inference_gpu_joules": gpu,
        # Power
        "avg_total_power_watts": total / time_seconds,
        "avg_cpu_power_watts": cpu / time_seconds,
        "avg_dram_power_watts": dram / time_seconds,
        "avg_gpu_power_watts": gpu / time_seconds,
        # Efficiency on the GPU energy only
        "accuracy_per_gpu_joule": accuracy / (gpu / n),
        "tokens_per_gpu_joule": tokens / gpu,
        "gpu_joules_per_token": gpu / tokens,
        "avg_gpu_joules_per_inference": gpu / n,
    }

## 5. Applying the correction

The cell below processes one run at a time: it loads the run, restores the recorded values, applies the correction and recomputes the summary. It also collects one row per inference and one row per run, which are used to validate the results in the next section.

Files are only rewritten when their content actually changes, and values that are already correct are left exactly as they are. Running the notebook on data that has already been corrected therefore changes nothing.

When `APPLY` is `True`, it writes:

- the per-prompt records of each subject (`<run>/<model>/CoT/all/<subject>.json`);
- the summary of each run (`<run>/run_data.json`);
- the run table (`run_table.csv`);
- a log of every corrected value (`rapl_corrections.csv`).

In [5]:
def same(a, b):
    return math.isclose(float(a), float(b), rel_tol=1e-12, abs_tol=1e-12)


inference_rows = []    # one row per inference
run_rows = []          # one row per run
overflow_log = []      # one row per corrected value
new_summaries = {}     # run name -> recomputed summary
files_changed = 0
summaries_changed = 0

for run_dir in run_directories():
    files, run_data = load_run(run_dir)
    records = [record for file_records in files.values() for record in file_records]
    if not records:
        print(f"[skip] {run_dir.name}: no per-prompt records")
        continue

    # Content currently on disk, to detect which files change
    on_disk = {path: json.dumps(file_records) for path, file_records in files.items()}

    # 1. Restore the recorded values
    for record in records:
        restore_recorded_values(record)
    recorded = {"cpu": sum(r[CPU_FIELD] for r in records), "total": sum(r[TOTAL_FIELD] for r in records)}
    recorded_cpu = [r[CPU_FIELD] for r in records]

    # 2. Reconstruct the energy lost to the counter overflow
    log, powers = apply_overflow_correction(records)
    for entry in log:
        overflow_log.append({"run": run_dir.name, **entry})

    # 3. Recompute the summary of the run
    summary = summarize_run(records, run_data)
    new_summaries[run_dir.name] = summary

    for record, recorded_joules in zip(records, recorded_cpu):
        inference_rows.append({
            "run": run_dir.name,
            "model": run_data["model"],
            "time": record[TIME_FIELD],
            "cpu_recorded": recorded_joules,
            "cpu": record[CPU_FIELD],
            "dram": record[DRAM_FIELD],
            "gpu": record[GPU_FIELD],
        })
    run_rows.append({
        "run": run_dir.name,
        "model": run_data["model"],
        "accuracy": run_data["avg_accuracy"],
        "cpu_reference_watts": powers["cpu"],
        "dram_reference_watts": powers["dram"],
        "recorded_cpu_joules": recorded["cpu"],
        "recorded_total_joules": recorded["total"],
        "time_seconds": sum(r[TIME_FIELD] for r in records),
        **summary,
    })

    # 4. Write the files that changed
    changed_files = [path for path, file_records in files.items() if json.dumps(file_records) != on_disk[path]]
    files_changed += len(changed_files)

    updated_run_data = dict(run_data)
    for key, value in summary.items():
        if key not in run_data or not same(run_data[key], value):
            updated_run_data[key] = value
    summary_changed = updated_run_data != run_data
    summaries_changed += summary_changed

    if APPLY:
        for path in changed_files:
            with open(path, "w") as f:
                json.dump(files[path], f)
        if summary_changed:
            with open(run_dir / "run_data.json", "w") as f:
                json.dump(updated_run_data, f, indent=4)

inferences = pd.DataFrame(inference_rows)
runs = pd.DataFrame(run_rows)
overflow_log = pd.DataFrame(overflow_log)

print(f"Runs processed: {len(runs)}   inferences: {len(inferences)}")
print(f"Per-prompt files that {'were' if APPLY else 'would be'} rewritten: {files_changed}")
print(f"run_data.json files that {'were' if APPLY else 'would be'} rewritten: {summaries_changed}")

Runs processed: 530   inferences: 197160
Per-prompt files that were rewritten: 0
run_data.json files that were rewritten: 530


In [6]:
# Run table: update the rows of every run, adding the new columns
run_table_path = RESULTS_DIR / "run_table.csv"
with open(run_table_path, "r", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = list(reader.fieldnames)
    rows = list(reader)

for key in next(iter(new_summaries.values())):
    if key not in fieldnames:
        fieldnames.append(key)

rows_changed = 0
for row in rows:
    summary = new_summaries.get(row["__run_id"])
    if summary is None:
        continue
    changed = False
    for key, value in summary.items():
        if row.get(key) in (None, "") or not same(row[key], value):
            row[key] = value
            changed = True
    rows_changed += changed

if APPLY:
    with open(run_table_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    overflow_log.to_csv(RESULTS_DIR / "rapl_corrections.csv", index=False)

print(f"run_table.csv rows that {'were' if APPLY else 'would be'} updated: {rows_changed} of {len(rows)}")
if not APPLY:
    print("\nAPPLY is False: nothing was written.")

run_table.csv rows that were updated: 530 of 530


## 6. Validation

### The lost energy was identified correctly

Three checks confirm that the lost multiples of $M$ were identified correctly:

- **Reference power.** It should be similar for every run, since the processor does the same work regardless of the model.
- **Distance to the expected energy.** For every corrected value, the distance to the expected energy is a small fraction of $M$. A wrong number of multiples would put it close to 1.
- **Power after the correction.** The power of every inference should be plausible. Before the correction, truncated values gave powers close to zero, and overflows gave powers of thousands of watts.

In [7]:
print("Reference power of the runs (W):")
print(runs[["cpu_reference_watts", "dram_reference_watts"]].describe().loc[["min", "50%", "max"]].round(2).to_string())

print(f"\nInferences corrected: {overflow_log.groupby(['run', 'question_id']).ngroups} of {len(inferences)}, in {overflow_log['run'].nunique()} of {len(runs)} runs")
print("\nNumber of values by lost multiples of M (k = -1: overflow during a short inference):")
print(overflow_log.groupby(["domain", "k"]).size().unstack(fill_value=0).to_string())

print("\nDistance of the corrected values to their expected energy, as a fraction of M:")
print(overflow_log["error_fraction"].describe().loc[["50%", "max"]].round(3).to_string())

timed = inferences[inferences["time"] >= 1]
power = pd.DataFrame({
    "recorded": timed["cpu_recorded"] / timed["time"],
    "corrected": timed["cpu"] / timed["time"],
})
print("\nPower of the first processor in each inference (W):")
print(power.describe().loc[["min", "50%", "max"]].round(1).to_string())

Reference power of the runs (W):
     cpu_reference_watts  dram_reference_watts
min                86.50                 11.81
50%                87.36                 11.96
max                98.89                 12.10

Inferences corrected: 24250 of 197160, in 228 of 530 runs

Number of values by lost multiples of M (k = -1: overflow during a short inference):
k       -1      1     2     3     4    5
domain                                  
cpu     17  14151  6571  1090  1836  568
dram    17      0     0     0     0    0

Distance of the corrected values to their expected energy, as a fraction of M:
50%    0.001
max    0.122

Power of the first processor in each inference (W):
     recorded  corrected
min       0.0       10.2
50%      87.2       87.4
max    4159.5      104.2


In [8]:
# Overflows that could not be detected leave a value about 150 J too low, at most once per counter range consumed
undetected = runs["total_inference_cpu_joules"] * 1e6 / MAX_ENERGY_RANGE_UJ
bound_joules = undetected.apply(math.ceil) * R_MOD_M / 1e6
print(f"Largest possible undetected error in a run: {bound_joules.max():.0f} J "
      f"({100 * (bound_joules / runs['total_inference_joules']).max():.3f}% of its total measured energy)")

Largest possible undetected error in a run: 2556 J (0.166% of its total measured energy)


### Effect on the results

The table below shows how much the total measured energy of each model changes with the correction. It mostly affects reasoning models, which have the longest inferences.

In [9]:
per_model = runs.groupby("model").agg(
    recorded=("recorded_total_joules", "sum"),
    corrected=("total_inference_joules", "sum"),
    gpu=("total_inference_gpu_joules", "sum"),
)
effect = pd.DataFrame({
    "change of the total (%)": 100 * (per_model["corrected"] / per_model["recorded"] - 1),
    "GPU share of the corrected total (%)": 100 * per_model["gpu"] / per_model["corrected"],
}).sort_values("change of the total (%)", ascending=False)

total_recorded, total_corrected = per_model["recorded"].sum(), per_model["corrected"].sum()
print(f"Total measured energy of the experiment: {total_recorded / 1e6:.0f} MJ recorded -> {total_corrected / 1e6:.0f} MJ corrected (+{100 * (total_corrected / total_recorded - 1):.1f}%)")
effect.round(1)

Total measured energy of the experiment: 1443 MJ recorded -> 1618 MJ corrected (+12.1%)


,change of the total (%),GPU share of the corrected total (%)
model,,
mistralai/Ministral-3-14B-Reasoning-2512,21.1,75.3
microsoft/Phi-4-reasoning,19.6,75.4
deepseek-ai/DeepSeek-R1-Distill-Qwen-14B,17.5,75.3
Qwen/Qwen3-14B (Thinking Mode),16.9,75.3
Qwen/Qwen3.5-9B (Thinking Mode),16.9,75.0
deepseek-ai/DeepSeek-R1-0528-Qwen3-8B,14.8,75.2
mistralai/Ministral-3-8B-Reasoning-2512,14.0,75.2
Qwen/Qwen3-8B (Thinking Mode),13.9,75.1
Qwen/Qwen3.5-4B (Thinking Mode),12.6,74.7


### GPU energy and total measured energy rank the models in the same way

The analysis uses the efficiency metrics based on the GPU energy, which was measured directly and needed no correction. The comparison below shows that this choice does not change the conclusions: ranking the models by accuracy per joule using the GPU energy, or using the total measured energy (GPU, first processor and its DRAM), gives almost the same order.

The first processor and its DRAM draw practically the same power for every model, which is why they add little more than a constant multiplied by the inference time.

In [10]:
by_model = runs.groupby("model")[["accuracy_per_gpu_joule", "accuracy_per_joule"]].mean()
gpu_rank = by_model["accuracy_per_gpu_joule"].rank(ascending=False)
total_rank = by_model["accuracy_per_joule"].rank(ascending=False)
difference = (gpu_rank - total_rank).abs()

print(f"Models: {len(by_model)}")
print(f"Models in a different position: {int((difference > 0).sum())}")
print(f"Largest change of position: {int(difference.max())}")
print(f"Correlation between the two rankings: {gpu_rank.corr(total_rank):.4f}")

# Power of the first processor and of its DRAM during inference, per model
power = runs.groupby("model")[["avg_cpu_power_watts", "avg_dram_power_watts"]].mean()
print("\nAverage power per model (W):")
print(power.agg(["min", "median", "max"]).round(1).to_string())

Models: 53
Models in a different position: 12
Largest change of position: 2
Correlation between the two rankings: 0.9993

Average power per model (W):
        avg_cpu_power_watts  avg_dram_power_watts
min                    86.5                  11.9
median                 87.3                  12.0
max                    89.3                  12.0
